# EutM — 1,2-Propanediol Docking to Crystal Structure (3I6P)

Docks 1,2-PD (12PD) to the pore of EutM, the BMC shell protein from the ethanolamine utilization (Eut) microcompartment.

- **Protein:** `3I6P_fixed.pdb` (EutM hexamer, chains A–F, derived from `3I6P.pdb1`)
- **Ligand:** `12PD_0001.pdb` / `12PD.params` (1,2-propanediol)
- **Pore residues:** 36–40 (Ile36–Gly37–Gly38–Gly39–Leu40) — 1 flanking residue each side of the GGG constriction, consistent with PduA/PduJ pore definitions
- **Output:** `EutM_Docking/`

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from math import sqrt

import pyrosetta
from pyrosetta import *
from pyrosetta.rosetta.core.pose import Pose
from pyrosetta.rosetta.core.import_pose import pose_from_file
from pyrosetta.rosetta.core.scoring import get_score_function, ScoreType, ScoreFunction
from pyrosetta.rosetta.core.scoring.constraints import AtomPairConstraint
from pyrosetta.rosetta.core.scoring.func import HarmonicFunc
from pyrosetta.rosetta.core.select.residue_selector import (
    ResidueIndexSelector, NeighborhoodResidueSelector,
    NotResidueSelector, AndResidueSelector,
)
from pyrosetta.rosetta.core.id import AtomID
from pyrosetta.rosetta.core.kinematics import MoveMap
from pyrosetta.rosetta.core.pack.task import TaskFactory
from pyrosetta.rosetta.core.pack.task.operation import (
    OperateOnResidueSubset, RestrictToRepackingRLT, PreventRepackingRLT,
    RestrictToRepacking,
)
from pyrosetta.rosetta.protocols.minimization_packing import MinMover, PackRotamersMover
from pyrosetta.rosetta.protocols.moves import MonteCarlo
from pyrosetta.rosetta.protocols.analysis import InterfaceAnalyzerMover
from pyrosetta.rosetta.protocols.relax import FastRelax
from pyrosetta.rosetta.protocols.rigid import RigidBodyPerturbMover, RigidBodyTransMover
from pyrosetta.rosetta.numeric import xyzVector_double_t

In [2]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
RAW_PDB           = "3I6P.pdb1"
FIXED_PDB         = "3I6P_fixed.pdb"
LIGAND_PDB        = "12PD_0001.pdb"
LIGAND_PARAMS     = "12PD.params"
OUT_DIR           = Path("EutM_Docking")

CHAINS            = ['A', 'B', 'C', 'D', 'E', 'F']

# EutM pore: Ile36–Gly37–Gly38–Gly39–Leu40
# Defined as 1 residue flanking the GGG motif on each side (5 residues total),
# consistent with PduA (Ile38–Gly39–Ser40–Gly41–Leu42) and PduJ definitions.
PORE_RESIDUE_RANGE = (36, 40)

# Docking parameters — identical to Propionaldehyde_Docking / Propionate_Docking
N_TRAJECTORIES    = 100
TRANSLATION_RANGE = 2.0    # Å perturbation per step
ROTATION_RANGE    = 45.0   # degrees

# Constraint: keep ligand near pore center during minimization/relax.
# 1,2-PD has affinity for non-pore regions of EutM; without these constraints
# the ligand drifts away from the pore during optimization.
# NOTE: constraints must NOT be included in interface ΔG scoring — the
# separated-state translation (1000 Å) causes massive harmonic penalty blowup.
CONSTRAINT_DISTANCE = 5.0   # Å
CONSTRAINT_STDEV    = 1.0   # Å
CONSTRAINT_WEIGHT   = 10.0

# MC temperature — consistent with Propionaldehyde/Propionate_Docking
MC_KT             = 1.0    # REU

# Flexibility (backbone stays rigid to preserve hexamer)
PORE_SIDECHAIN_FLEX = True
BINDING_SITE_FLEX   = True

OUT_DIR.mkdir(exist_ok=True)

pyrosetta.init(f"-mute all -ex1 -ex2")
sf = get_score_function()
print("PyRosetta initialized.")
print(f"Pore residues: {PORE_RESIDUE_RANGE[0]}–{PORE_RESIDUE_RANGE[1]} (Ile36–Gly37–Gly38–Gly39–Leu40)")
print(f"MC kT: {MC_KT}")

┌───────────────────────────────────────────────────────────────────────────────┐
│                                  PyRosetta-4                                  │
│               Created in JHU by Sergey Lyskov and PyRosetta Team              │
│               (C) Copyright Rosetta Commons Member Institutions               │
│                                                                               │
│ NOTE: USE OF PyRosetta FOR COMMERCIAL PURPOSES REQUIRES PURCHASE OF A LICENSE │
│          See LICENSE.PyRosetta.md or email license@uw.edu for details         │
└───────────────────────────────────────────────────────────────────────────────┘
PyRosetta-4 2026 [Rosetta PyRosetta4.Release.python311.m1 2026.03+releasequarterly.5e498f1409c68ade56c8ce5842bf79e1b02e8db4 2026-01-13T13:24:11] retrieved from: http://www.pyrosetta.org
PyRosetta initialized.
Pore residues: 36–40 (Ile36–Gly37–Gly38–Gly39–Leu40)
MC kT: 1.0


In [3]:
# ── STRUCTURE PREPARATION ─────────────────────────────────────────────────────
# We strip HETATM (SO4 crystallographic sulfates) and alternate conformations
# (ACYS/BCYS at 41, AMET/BMET at 44) before loading into PyRosetta.

def prepare_eutm_pdb(in_pdb, out_pdb):
    """Strip HETATM and keep only alt-A (or no-alt) ATOM records."""
    kept = []
    with open(in_pdb) as fh:
        for line in fh:
            record = line[:6].strip()
            if record == "HETATM":
                continue  # drop SO4 and other small molecules
            if record == "ATOM":
                alt_loc = line[16]  # column 17 (0-indexed: 16)
                if alt_loc not in (' ', 'A'):
                    continue  # drop alt-B and higher conformations
                # Blank out the alt-loc field so PyRosetta doesn't complain
                line = line[:16] + ' ' + line[17:]
            kept.append(line)
    with open(out_pdb, 'w') as fh:
        fh.writelines(kept)
    print(f"  Written: {out_pdb}  ({sum(1 for l in kept if l.startswith('ATOM'))} ATOM records)")

if not Path(FIXED_PDB).exists():
    print(f"Preparing {FIXED_PDB} from {RAW_PDB}...")
    prepare_eutm_pdb(RAW_PDB, FIXED_PDB)
else:
    print(f"Using existing {FIXED_PDB}")

# Verify chains
chains_seen = set()
with open(FIXED_PDB) as fh:
    for line in fh:
        if line.startswith("ATOM"):
            chains_seen.add(line[21])
print(f"Chains in fixed PDB: {sorted(chains_seen)}")

Using existing 3I6P_fixed.pdb
Chains in fixed PDB: ['A', 'B', 'C', 'D', 'E', 'F']


In [4]:
# ── HELPER FUNCTIONS ──────────────────────────────────────────────────────────

def calculate_interface_energy_iam(pose, sf_score):
    """
    Interface dG via InterfaceAnalyzerMover (pack_separated=True).

    Matches calculate_interface_energy_iam() in Propionaldehyde_Docking.ipynb
    and Propionate_Docking.ipynb exactly.  sf_score must carry NO constraint
    weight — the harmonic penalty on a 1000 Å separated-state translation
    causes a ~300 M REU blowup.  pack_separated=True repacks the interface
    in the separated state so E_sep is physically meaningful.
    """
    iam = InterfaceAnalyzerMover(
        pose.num_jump(),   # last jump = ligand rigid-body jump
        False,             # tracer output
        sf_score,          # constraint-free ref2015
        False,             # compute_packstat
        False,             # pack_input
        True,              # pack_separated
        False,             # use_jobname
    )
    iam.apply(pose)
    E_complex   = sf_score(pose)
    interface   = iam.get_interface_dG()
    E_separated = E_complex - interface
    return {'total': E_complex, 'separated': E_separated, 'interface': interface}


def make_interface_pack_mover(pose, ligand_resnum, sf, radius=6.0):
    """
    PackRotamersMover restricted to protein residues within radius of the ligand.
    Applied post-perturbation to relieve steric clashes before gradient
    minimization — mirrors Rosetta DockingHighRes protocol and matches
    make_interface_pack_mover() in Propionaldehyde/Propionate_Docking.ipynb.
    """
    ligand_sel    = ResidueIndexSelector(str(ligand_resnum))
    interface_sel = NeighborhoodResidueSelector()
    interface_sel.set_focus_selector(ligand_sel)
    interface_sel.set_distance(radius)
    interface_sel.set_include_focus_in_subset(False)

    tf = TaskFactory()
    tf.push_back(OperateOnResidueSubset(RestrictToRepackingRLT(), interface_sel))
    tf.push_back(OperateOnResidueSubset(PreventRepackingRLT(), interface_sel, True))

    packer = PackRotamersMover()
    packer.score_function(sf)
    packer.task_factory(tf)
    return packer


def get_ligand_rmsd(pose1, pose2, ligand_res):
    """Heavy-atom RMSD of the ligand between two poses."""
    lig1, lig2 = pose1.residue(ligand_res), pose2.residue(ligand_res)
    sq, n = 0.0, 0
    for i in range(1, min(lig1.natoms(), lig2.natoms()) + 1):
        if not lig1.atom_is_hydrogen(i):
            d = lig1.xyz(i).distance(lig2.xyz(i))
            sq += d * d
            n += 1
    return (sq / n) ** 0.5 if n else 0.0


def get_pore_rmsd(pose1, pose2, pore_residues):
    """Backbone RMSD of pore residues (N/CA/C/O) between two poses."""
    sq, n = 0.0, 0
    for r in pore_residues:
        r1, r2 = pose1.residue(r), pose2.residue(r)
        for atom in ('N', 'CA', 'C', 'O'):
            if r1.has(atom) and r2.has(atom):
                d = r1.xyz(r1.atom_index(atom)).distance(r2.xyz(r2.atom_index(atom)))
                sq += d * d
                n += 1
    return (sq / n) ** 0.5 if n else 0.0


def get_min_distance_to_pore(pose, ligand_resnum, pore_residues):
    """Minimum heavy-atom distance from ligand to any pore residue atom."""
    ligand = pose.residue(ligand_resnum)
    min_d = 999.0
    for pr in pore_residues:
        pres = pose.residue(pr)
        for i in range(1, ligand.natoms() + 1):
            if ligand.atom_is_hydrogen(i):
                continue
            for j in range(1, pres.natoms() + 1):
                if pres.atom_is_hydrogen(j):
                    continue
                min_d = min(min_d, ligand.xyz(i).distance(pres.xyz(j)))
    return min_d


def preserve_chain_info(src, tgt):
    """Copy chain IDs from src pose to tgt pose."""
    if src.pdb_info() and tgt.pdb_info():
        for i in range(1, min(src.total_residue(), tgt.total_residue()) + 1):
            tgt.pdb_info().chain(i, src.pdb_info().chain(i))


def add_pore_constraints(pose, ligand_resnum, pore_residues, target_dist, stdev):
    """Atom-pair constraints from the central ligand heavy atom to each pore CA."""
    ligand = pose.residue(ligand_resnum)
    heavy = [i for i in range(1, ligand.natoms() + 1)
             if not ligand.atom_is_hydrogen(i)]
    if not heavy:
        raise RuntimeError("Ligand has no heavy atoms.")
    lig_atom = heavy[len(heavy) // 2]
    lig_aid  = AtomID(lig_atom, ligand_resnum)
    func = HarmonicFunc(target_dist, stdev)
    n = 0
    for pr in pore_residues:
        pres = pose.residue(pr)
        if not pres.has("CA"):
            continue
        pore_aid = AtomID(pres.atom_index("CA"), pr)
        pose.add_constraint(AtomPairConstraint(lig_aid, pore_aid, func))
        n += 1
    print(f"  Added {n} pore constraints (target {target_dist} ± {stdev} Å)")
    return lig_atom


print("Helper functions defined.")

Helper functions defined.


In [5]:
# ── LOAD STRUCTURES ───────────────────────────────────────────────────────────
print("Loading EutM hexamer...")
protein_pose = Pose()
pose_from_file(protein_pose, FIXED_PDB)
print(f"  Residues: {protein_pose.total_residue()}")
print(f"  Chains:   {protein_pose.num_chains()}")

# Report chain layout
chain_map = {}
for i in range(1, protein_pose.total_residue() + 1):
    if protein_pose.pdb_info():
        c = protein_pose.pdb_info().chain(i)
        chain_map.setdefault(c, []).append(protein_pose.pdb_info().number(i))
for c, nums in sorted(chain_map.items()):
    print(f"  Chain {c}: residues {nums[0]}–{nums[-1]} ({len(nums)} res)")

print("\nLoading 1,2-PD ligand...")
ligand_pose = Pose()
pose_from_file(ligand_pose, LIGAND_PDB)
print(f"  Ligand residues: {ligand_pose.total_residue()}")
print(f"  Ligand type: {ligand_pose.residue(1).name()}")

Loading EutM hexamer...
  Residues: 540
  Chains:   6
  Chain A: residues 1–90 (90 res)
  Chain B: residues 1–90 (90 res)
  Chain C: residues 1–90 (90 res)
  Chain D: residues 1–90 (90 res)
  Chain E: residues 1–90 (90 res)
  Chain F: residues 1–90 (90 res)

Loading 1,2-PD ligand...
  Ligand residues: 1
  Ligand type: 12PD


In [6]:
# ── IDENTIFY PORE RESIDUES ────────────────────────────────────────────────────
print("Identifying pore residues (PDB numbers {0}–{1})...".format(*PORE_RESIDUE_RANGE))

pore_residues = []
for i in range(1, protein_pose.total_residue() + 1):
    if not protein_pose.pdb_info():
        continue
    chain  = protein_pose.pdb_info().chain(i)
    pdbnum = protein_pose.pdb_info().number(i)
    if chain in CHAINS and PORE_RESIDUE_RANGE[0] <= pdbnum <= PORE_RESIDUE_RANGE[1]:
        pore_residues.append(i)
        resname = protein_pose.residue(i).name1()
        print(f"  {chain}{pdbnum} ({resname}) → pose residue {i}")

print(f"\nTotal pore residues: {len(pore_residues)}")

# Pore center
pore_coords = np.array([
    [protein_pose.residue(r).xyz(protein_pose.residue(r).atom_index("CA")).x,
     protein_pose.residue(r).xyz(protein_pose.residue(r).atom_index("CA")).y,
     protein_pose.residue(r).xyz(protein_pose.residue(r).atom_index("CA")).z]
    for r in pore_residues if protein_pose.residue(r).has("CA")
])
pore_center = pore_coords.mean(axis=0)
print(f"Pore center: ({pore_center[0]:.2f}, {pore_center[1]:.2f}, {pore_center[2]:.2f}) Å")

Identifying pore residues (PDB numbers 36–40)...
  A36 (I) → pose residue 36
  A37 (G) → pose residue 37
  A38 (G) → pose residue 38
  A39 (G) → pose residue 39
  A40 (L) → pose residue 40
  B36 (I) → pose residue 126
  B37 (G) → pose residue 127
  B38 (G) → pose residue 128
  B39 (G) → pose residue 129
  B40 (L) → pose residue 130
  C36 (I) → pose residue 216
  C37 (G) → pose residue 217
  C38 (G) → pose residue 218
  C39 (G) → pose residue 219
  C40 (L) → pose residue 220
  D36 (I) → pose residue 306
  D37 (G) → pose residue 307
  D38 (G) → pose residue 308
  D39 (G) → pose residue 309
  D40 (L) → pose residue 310
  E36 (I) → pose residue 396
  E37 (G) → pose residue 397
  E38 (G) → pose residue 398
  E39 (G) → pose residue 399
  E40 (L) → pose residue 400
  F36 (I) → pose residue 486
  F37 (G) → pose residue 487
  F38 (G) → pose residue 488
  F39 (G) → pose residue 489
  F40 (L) → pose residue 490

Total pore residues: 30
Pore center: (17.92, -31.91, 26.89) Å


In [7]:
# ── BUILD COMPLEX & POSITION LIGAND AT PORE ───────────────────────────────────
print("Building protein–ligand complex...")
pose = protein_pose.clone()
pose.append_residue_by_jump(ligand_pose.residue(1), pose.total_residue())
ligand_resnum = pose.total_residue()
if pose.pdb_info():
    pose.pdb_info().chain(ligand_resnum, 'L')
print(f"  Ligand appended as residue {ligand_resnum} (chain L)")

# Translate ligand centroid → pore center
lig = pose.residue(ligand_resnum)
lig_xyz = np.array([[lig.xyz(i).x, lig.xyz(i).y, lig.xyz(i).z]
                    for i in range(1, lig.natoms() + 1)])
lig_center = lig_xyz.mean(axis=0)
delta = pore_center - lig_center
t_axis = xyzVector_double_t(float(delta[0]), float(delta[1]), float(delta[2]))

trans = RigidBodyTransMover(pose, pose.num_jump())
trans.trans_axis(t_axis)
trans.apply(pose)

min_d = get_min_distance_to_pore(pose, ligand_resnum, pore_residues)
print(f"  Min ligand–pore distance after centering: {min_d:.2f} Å")

initial_pdb = str(OUT_DIR / "00_initial_complex.pdb")
pose.dump_pdb(initial_pdb)
print(f"  Saved: {initial_pdb}")

Building protein–ligand complex...
  Ligand appended as residue 541 (chain L)
  Min ligand–pore distance after centering: 33.53 Å
  Saved: EutM_Docking/00_initial_complex.pdb


In [8]:
# ── CONSTRAINTS & SCORE FUNCTIONS ─────────────────────────────────────────────
# Constraints keep 1,2-PD in the pore during minimization/relax.
# 1,2-PD has affinity for non-pore regions of EutM; without these it drifts.
# sf_soft / sf_hard are used ONLY for optimization — NOT for interface scoring.
# Interface ΔG is always calculated with plain sf (no constraint weight) so
# the 1000 Å separated-state translation does not incur a harmonic blowup.
print("Adding pore constraints...")
add_pore_constraints(pose, ligand_resnum, pore_residues,
                     CONSTRAINT_DISTANCE, CONSTRAINT_STDEV)

sf_soft = ScoreFunction()
sf_soft.set_weight(ScoreType.fa_atr,    0.2)
sf_soft.set_weight(ScoreType.fa_rep,    0.1)
sf_soft.set_weight(ScoreType.fa_sol,    0.2)
sf_soft.set_weight(ScoreType.hbond_sc,  1.0)
sf_soft.set_weight(ScoreType.hbond_bb_sc, 1.0)
sf_soft.set_weight(ScoreType.atom_pair_constraint, CONSTRAINT_WEIGHT)

sf_hard = get_score_function()
sf_hard.set_weight(ScoreType.atom_pair_constraint, CONSTRAINT_WEIGHT)
print("  Score functions ready.")

Adding pore constraints...
  Added 30 pore constraints (target 5.0 ± 1.0 Å)
  Score functions ready.


In [9]:
# ── PORE SIDECHAIN PREP ────────────────────────────────────────────────────────
# Pack pore residues + 10 Å neighborhood before docking to relieve
# crystal-packing artifacts at the binding site.  Backbone is frozen
# (RestrictToRepacking).  Matches Propionaldehyde/Propionate_Docking.ipynb.
print("Pore sidechain prep (10 Å neighborhood repacking)...")
pore_sel = ResidueIndexSelector(','.join(str(r) for r in pore_residues))
nbr_sel  = NeighborhoodResidueSelector(pore_sel, distance=10.0,
                                        include_focus_in_subset=True)
tf = TaskFactory()
tf.push_back(RestrictToRepacking())
tf.push_back(OperateOnResidueSubset(PreventRepackingRLT(), nbr_sel, flip_subset=True))

packer = PackRotamersMover(sf_hard)
packer.task_factory(tf)
packer.apply(pose)
print(f"  Packed {len(pore_residues)} pore residues + 10 Å neighborhood.")

Pore sidechain prep (10 Å neighborhood repacking)...
  Packed 30 pore residues + 10 Å neighborhood.


In [10]:
# ── MOVE MAP ──────────────────────────────────────────────────────────────────
movemap = MoveMap()
movemap.set_bb(False)    # backbone RIGID — preserve hexamer geometry
movemap.set_chi(False)
movemap.set_jump(False)

if PORE_SIDECHAIN_FLEX:
    for r in pore_residues:
        movemap.set_chi(r, True)
    print(f"Pore sidechains flexible ({len(pore_residues)} residues).")

if BINDING_SITE_FLEX:
    # Use setter-based construction to avoid positional arg ambiguity.
    bs_sel = NeighborhoodResidueSelector()
    bs_sel.set_focus_selector(ResidueIndexSelector(str(ligand_resnum)))
    bs_sel.set_distance(10.0)
    bs_sel.set_include_focus_in_subset(False)
    bs_mask = bs_sel.apply(pose)
    bs_count = 0
    for i in range(1, pose.total_residue() + 1):
        if bs_mask[i] and i != ligand_resnum:
            movemap.set_chi(i, True)
            bs_count += 1
    print(f"Binding-site sidechains flexible ({bs_count} residues, 10 Å shell).")

jump_num = pose.num_jump()
movemap.set_jump(jump_num, True)
print(f"Ligand rigid-body jump {jump_num} enabled.")
print("Hexamer backbone: RIGID.")

Pore sidechains flexible (30 residues).
Binding-site sidechains flexible (0 residues, 10 Å shell).
Ligand rigid-body jump 6 enabled.
Hexamer backbone: RIGID.


In [11]:
# ── MOVERS ────────────────────────────────────────────────────────────────────
perturb_trans = RigidBodyPerturbMover(jump_num, TRANSLATION_RANGE, 0.0)
perturb_rot   = RigidBodyPerturbMover(jump_num, 0.0, ROTATION_RANGE)

min_soft = MinMover()
min_soft.movemap(movemap)
min_soft.score_function(sf_soft)
min_soft.min_type("lbfgs_armijo_nonmonotone")
min_soft.tolerance(0.01)

min_hard = MinMover()
min_hard.movemap(movemap)
min_hard.score_function(sf_hard)
min_hard.min_type("lbfgs_armijo_nonmonotone")
min_hard.tolerance(0.001)

relax = FastRelax()
relax.set_scorefxn(sf_hard)
relax.set_movemap(movemap)
# NOTE: max_iter() intentionally NOT called — calling it converts FastRelax
# into a plain minimizer by bypassing the alternating pack↔minimize cycles
# with score-weight ramping that give FastRelax its structural quality.

# Interface repack mover: relieve steric clashes post-perturbation, before
# gradient minimization — 6 Å shell, matching Propionaldehyde/Propionate_Docking.
interface_pack = make_interface_pack_mover(pose, ligand_resnum, sf_hard, radius=6.0)

print(f"Perturbation:     ±{TRANSLATION_RANGE} Å, ±{ROTATION_RANGE}°")
print(f"Interface repack: 6 Å shell (post-perturbation, pre-minimization)")
print(f"Soft min:         sf_soft, tol 0.01  ×3")
print(f"Hard min:         sf_hard, tol 0.001 ×2")
print(f"FastRelax:        sf_hard (full weight ramping)")

Perturbation:     ±2.0 Å, ±45.0°
Interface repack: 6 Å shell (post-perturbation, pre-minimization)
Soft min:         sf_soft, tol 0.01  ×3
Hard min:         sf_hard, tol 0.001 ×2
FastRelax:        sf_hard (full weight ramping)


In [ ]:
# ── DOCKING RUN ───────────────────────────────────────────────────────────────
# Protocol now matches Propionaldehyde_Docking / Propionate_Docking exactly:
#   MC walk (kT=1.0): perturb → interface repack (6 Å) → soft-min ×3
#                     → hard-min ×2 → FastRelax → Metropolis acceptance
#   Interface ΔG via InterfaceAnalyzerMover (pack_separated=True, constraint-free sf)
print(f"Running {N_TRAJECTORIES} MC docking trajectories (kT={MC_KT})...")
results     = []
native_pose = pose.clone()
mc          = MonteCarlo(native_pose, sf_hard, MC_KT)

for i in range(1, N_TRAJECTORIES + 1):
    if i % 20 == 0:
        print(f"  {i}/{N_TRAJECTORIES}")

    test = mc.last_accepted_pose().clone()

    # Rigid-body perturbation
    perturb_trans.apply(test)
    perturb_rot.apply(test)

    # Repack interface to relieve steric clashes before gradient minimization
    interface_pack.apply(test)

    # Minimize: soft repulsion (clash relief) → full atom → FastRelax
    for _ in range(3):
        min_soft.apply(test)
    for _ in range(2):
        min_hard.apply(test)
    relax.apply(test)

    # Metropolis criterion
    accepted = mc.boltzmann(test)
    preserve_chain_info(native_pose, test)

    # Score interface ΔG with constraint-free sf via IAM (pack_separated=True)
    energies = calculate_interface_energy_iam(test, sf)
    results.append({
        'model':             i,
        'protein':           'EutM',
        'total_energy':      energies['total'],
        'separated_energy':  energies['separated'],
        'interface_energy':  energies['interface'],
        'ligand_rmsd':       get_ligand_rmsd(test, native_pose, ligand_resnum),
        'pore_rmsd':         get_pore_rmsd(test, native_pose, pore_residues),
        'min_pore_distance': get_min_distance_to_pore(test, ligand_resnum, pore_residues),
        'mc_accepted':       accepted,
    })
    test.dump_pdb(str(OUT_DIR / f"dock_{i:04d}.pdb"))

print("Done.")

Running 100 MC docking trajectories (kT=1.0)...


In [ ]:
# ── RESULTS ───────────────────────────────────────────────────────────────────
df = pd.DataFrame(results).sort_values('interface_energy').reset_index(drop=True)
df.to_csv(OUT_DIR / "docking_scores.csv", index=False)
df.head(10).to_csv(OUT_DIR / "top10_scores.csv", index=False)

print("="*65)
print(" EutM — 1,2-PD DOCKING SUMMARY")
print("="*65)
print(f"Models generated:  {len(df)}")
print(f"Best interface ΔG: {df['interface_energy'].min():.2f} REU")
print(f"Mean interface ΔG: {df['interface_energy'].mean():.2f} REU")
print(f"Top-10 mean ΔG:    {df.head(10)['interface_energy'].mean():.2f} REU")
print(f"MC accept rate:    {df['mc_accepted'].mean()*100:.1f}%")
print(f"Mean pore RMSD:    {df['pore_rmsd'].mean():.2f} Å  (hexamer preservation)")
print(f"Min pore distance: {df['min_pore_distance'].min():.2f} Å")
print()
print(f"Best model: {OUT_DIR}/dock_{df.iloc[0]['model']:04.0f}.pdb")
print(f"Scores:     {OUT_DIR}/docking_scores.csv")

# Plot energy funnel
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(df['ligand_rmsd'], df['interface_energy'],
                alpha=0.7, s=30, color='steelblue')
axes[0].set_xlabel('Ligand RMSD from start (Å)')
axes[0].set_ylabel('Interface ΔG (REU)')
axes[0].set_title('EutM 1,2-PD Docking Funnel')

axes[1].scatter(df['min_pore_distance'], df['interface_energy'],
                alpha=0.7, s=30, color='darkorange')
axes[1].axvline(CONSTRAINT_DISTANCE, color='red', lw=1.2, linestyle='--',
                label=f'Target {CONSTRAINT_DISTANCE} Å')
axes[1].set_xlabel('Min ligand–pore distance (Å)')
axes[1].set_ylabel('Interface ΔG (REU)')
axes[1].set_title('Pore Proximity vs Interface Energy')
axes[1].legend()

plt.tight_layout()
plt.savefig(OUT_DIR / "docking_funnel.png", dpi=150, bbox_inches='tight')
plt.show()

print("\nTop 10 models:")
print("-"*75)
print(f"{'Rank':>4}  {'Model':>5}  {'Interface':>10}  {'Min→Pore':>9}  "
      f"{'Pore RMSD':>10}  {'MC':>5}")
print("-"*75)
for rank, row in df.head(10).iterrows():
    print(f"{rank+1:>4}  {row['model']:>5.0f}  {row['interface_energy']:>10.2f}  "
          f"{row['min_pore_distance']:>9.2f}  {row['pore_rmsd']:>10.2f}  "
          f"{'Y' if row['mc_accepted'] else 'N':>5}")